In [2]:
from neo4j import GraphDatabase
import pandas as pd

# 데이터 로딩
df = pd.read_csv("청년정책목록_전체.csv")

# Neo4j 연결 설정
uri = "bolt://localhost:7687"  # 또는 AuraDB 주소
user = "neo4j"
password = "neo4j1234"

driver = GraphDatabase.driver(uri, auth=(user, password))

def create_nodes_and_relationships(tx, row):
    # 정책 노드
    tx.run("""
        MERGE (p:Policy {id: $plcyNo})
        SET p.name = $plcyNm, p.description = $plcyExplnCn
    """, plcyNo=row['plcyNo'], plcyNm=row['plcyNm'], plcyExplnCn=row['plcyExplnCn'])

    # 제공 기관
    if pd.notna(row['sprvsnInstCdNm']):
        tx.run("""
            MERGE (o:Organization {name: $org})
            MERGE (p:Policy {id: $plcyNo})-[:PROVIDED_BY]->(o)
        """, org=row['sprvsnInstCdNm'], plcyNo=row['plcyNo'])

    # 카테고리
    if pd.notna(row['mclsfNm']):
        tx.run("""
            MERGE (c:Category {name: $category})
            MERGE (p:Policy {id: $plcyNo})-[:BELONGS_TO]->(c)
        """, category=row['mclsfNm'], plcyNo=row['plcyNo'])

    # 연령대
    if pd.notna(row['sprtTrgtMinAge']) and pd.notna(row['sprtTrgtMaxAge']):
        age_range = f"{int(row['sprtTrgtMinAge'])}-{int(row['sprtTrgtMaxAge'])}"
        tx.run("""
            MERGE (a:AgeGroup {range: $range})
            MERGE (p:Policy {id: $plcyNo})-[:TARGETS]->(a)
        """, range=age_range, plcyNo=row['plcyNo'])

    # 키워드
    if pd.notna(row['plcyKywdNm']):
        keywords = str(row['plcyKywdNm']).split(',')
        for kw in keywords:
            kw = kw.strip()
            tx.run("""
                MERGE (k:Keyword {name: $keyword})
                MERGE (p:Policy {id: $plcyNo})-[:HAS_KEYWORD]->(k)
            """, keyword=kw, plcyNo=row['plcyNo'])

    # 지역
    if pd.notna(row['rgtrHghrkInstCdNm']):
        tx.run("""
            MERGE (r:Region {name: $region})
            MERGE (p:Policy {id: $plcyNo})-[:AVAILABLE_IN]->(r)
        """, region=row['rgtrHghrkInstCdNm'], plcyNo=row['plcyNo'])

# 삽입 실행
with driver.session() as session:
    for idx, row in df.iterrows():
        session.write_transaction(create_nodes_and_relationships, row)

driver.close()



C:\Users\Playdata\AppData\Local\Temp\ipykernel_23676\1529868676.py:63: DeprecationWarning: write_transaction has been renamed to execute_write
  session.write_transaction(create_nodes_and_relationships, row)
